In [232]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score, median_absolute_error
from sklearn.impute import KNNImputer

In [233]:
df = pd.read_csv("dataset.csv")

In [234]:
print(df.head())
print(df.info())
print(df.isnull().sum())

   User_ID  Heart_Rate_(BPM)  Blood_Oxygen_Level_(%)    Step_Count  \
0   4174.0         58.939776               98.809650   5450.390578   
1      NaN               NaN               98.532195    727.601610   
2   1860.0        247.803052               97.052954   2826.521994   
3   2294.0         40.000000               96.894213  13797.338044   
4   2130.0         61.950165               98.583797  15679.067648   

  Sleep_Duration_(hours) Activity_Level Stress_Level  
0      7.167235622316564  Highly Active            1  
1      6.538239375570314  Highly_Active            5  
2                  ERROR  Highly Active            5  
3      7.367789630207228          Actve            3  
4                    NaN  Highly_Active            6  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   User_ID                 9799 no

In [235]:
df.describe()

,User_ID,Heart_Rate_(BPM),Blood_Oxygen_Level_(%),Step_Count
count,9799.000000,9600.000000,9700.000000,9900.000000
mean,3007.480253,76.035462,97.841581,6985.685885
std,1150.581542,19.412483,1.732863,6885.809680
min,1001.000000,40.000000,90.791208,0.910138
25%,1997.500000,64.890152,96.662683,2021.039657
50%,2998.000000,75.220601,98.010642,4962.534599
75%,4004.000000,85.198249,99.376179,9724.902880
max,4999.000000,296.593970,100.000000,62486.690753


In [217]:
df.shape

(10000, 7)

In [218]:
df.columns


Index(['User_ID', 'Heart_Rate_(BPM)', 'Blood_Oxygen_Level_(%)', 'Step_Count',
       'Sleep_Duration_(hours)', 'Activity_Level', 'Stress_Level'],
      dtype='object')

In [236]:
print(df['User_ID'].value_counts(dropna=False))


User_ID
NaN       201
3708.0     10
3773.0     10
1018.0      9
1661.0      9
         ... 
4142.0      1
3858.0      1
2584.0      1
2163.0      1
4887.0      1
Name: count, Length: 3635, dtype: int64


In [237]:
#User_ID sütununun index işlevini yerine getirmediği unique olması gerekirken tekrar eden veriler içerdiğini 
#görebiliriz ve çok fazla boş değer var.Bu sütunu siliyoruz.Eğer aynı değerler aynı kullanıcının verileri olarak yorumlanırsa
#bu sütundaki boş yerelre değerler atanıp sütun silinmeyebilir

df.drop('User_ID', axis=1, inplace=True)

In [238]:
numeric_columns = [
       
        'Heart_Rate_(BPM)', 
        'Blood_Oxygen_Level_(%)', 
        'Step_Count', 
        'Sleep_Duration_(hours)', 
        'Stress_Level'
    ]


# 1. Tüm string değerleri NaN'a çevirme
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

def clean_step_count(series):
    # String ve geçersiz değerleri temizle
    series = pd.to_numeric(series, errors='coerce')
    # Negatif değerleri ve aşırı büyük değerleri temizle (örn. >50,000)
    series = series.mask((series < 0) | (series > 50000) | (series==40))
    # Float'tan Int64'e çevir (NaN'ları koruyarak)
    return series.round().astype('Int64')

df['Heart_Rate_(BPM)'] = df['Heart_Rate_(BPM)'].replace(40.0, np.nan)



# 2. Mantıksız değerleri temizleme fonksiyonu BUNU NEDEN YAPTIĞIMIZ VERİ SETİNDEKİ UNİQUE DEĞERLER PRİNT EDİELREK GÖSTERİLECEK
def clean_numeric_values(series, min_val, max_val):
    """Belirtilen aralık dışındaki değerleri NULL yapar"""
    return series.where((series >= min_val) & (series <= max_val))

# 3. Her sütun için mantık kurallarını uygula
df['Heart_Rate_(BPM)'] = clean_numeric_values(df['Heart_Rate_(BPM)'], 20, 150)  # Normal nabız aralığı

df['Blood_Oxygen_Level_(%)'] = clean_numeric_values(df['Blood_Oxygen_Level_(%)'], 70, 100)  # Oksijen seviyesi
df['Step_Count'] = clean_numeric_values(df['Step_Count'], 0, 100000)  # Makul adım sayısı
df['Step_Count'] = clean_step_count(df['Step_Count'])  # Makul adım sayısı
df['Sleep_Duration_(hours)'] = clean_numeric_values(df['Sleep_Duration_(hours)'], 0, 24)  # 0-24 saat
df['Stress_Level'] = clean_numeric_values(df['Stress_Level'], 0, 10)  # 0-10 arası stres seviyesi




# 4. Adım sayısını integer'a yuvarla
df['Step_Count'] = df['Step_Count'].round().astype('Int64')  # Int64 NaN'leri destekler

# 5. Temizlenmiş veriyi göster
print("Temizlenmiş veri istatistikleri:")
print(df[numeric_columns].describe())

# 6. Eksik veri sayısını göster
print("\nEksik veri sayısı:")
print(df[numeric_columns].isnull().sum())
print(df.isnull().sum())
df.head()

Temizlenmiş veri istatistikleri:
       Heart_Rate_(BPM)  Blood_Oxygen_Level_(%)   Step_Count  \
count       9466.000000             9700.000000       9891.0   
mean          75.444447               97.841581  6940.271661   
std           14.458886                1.732863  6720.840127   
min           40.121512               90.791208          1.0   
25%           65.108097               96.662683       2019.5   
50%           75.265638               98.010642       4955.0   
75%           85.121403               99.376179       9715.5   
max          130.312002              100.000000      49119.0   

       Sleep_Duration_(hours)  Stress_Level  
count             9602.000000   9751.000000  
mean                 6.506161      5.464055  
std                  1.508558      2.871561  
min                  0.589987      1.000000  
25%                  5.494408      3.000000  
50%                  6.503462      5.000000  
75%                  7.529543      8.000000  
max                 12

,Heart_Rate_(BPM),Blood_Oxygen_Level_(%),Step_Count,Sleep_Duration_(hours),Activity_Level,Stress_Level
0,58.939776,98.809650,5450,7.167236,Highly Active,1.0
1,NaN,98.532195,728,6.538239,Highly_Active,5.0
2,NaN,97.052954,2827,NaN,Highly Active,5.0
3,NaN,96.894213,13797,7.367790,Actve,3.0
4,61.950165,98.583797,15679,NaN,Highly_Active,6.0


In [239]:

from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler

def custom_knn_imputation(df, n_neighbors=3):
    """
    Sayısal sütunlardaki eksik verileri KNN ile doldurur.
    - Kategorik veriler etkilenmez.
    - Orijinal veri tipleri korunur.
    - NaN içeren hücreler hariç int dönüşümü yapılır.
    - Fiziksel sınırlar uygulanır.
    """
    
    # Sayısal sütunları tanımla (User ID hariç)
    numeric_columns = [
        'Heart_Rate_(BPM)', 
        'Blood_Oxygen_Level_(%)', 
        'Step_Count', 
        'Sleep_Duration_(hours)', 
        'Stress_Level'
    ]
    
    # Sadece DataFrame'de mevcut olan sütunları seç
    numeric_columns = [col for col in numeric_columns if col in df.columns]
    
    if not numeric_columns:
        print("Uyarı: İşlenecek sayısal sütun bulunamadı.")
        return

    # Sayısal sütunlardaki eksik değerleri kontrol et
    null_mask = df[numeric_columns].isnull().any(axis=1)
    if not null_mask.any():
        print("Uyarı: Hiç null değer yok.")
        return

    # Orijinal veri tiplerini sakla
    col_dtypes = {col: df[col].dtype for col in numeric_columns}

    # Eksik verili satırları seç
    null_indices = df.index[null_mask]
    null_data = df.loc[null_indices, numeric_columns].copy()

    # MinMax ölçekleme
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(null_data)

    # KNN Imputation
    imputer = KNNImputer(n_neighbors=min(n_neighbors, len(df)-1))
    imputed_data = imputer.fit_transform(scaled_data)

    # Orijinal ölçeğe dön
    rescaled_data = scaler.inverse_transform(imputed_data)

    # NaN'leri yalnızca ihtiyaç duyulan hücrelerde güncelle
    for i, col in enumerate(numeric_columns):
        null_col_mask = df.loc[null_indices, col].isnull()
        imputed_values = rescaled_data[:, i]

        if pd.api.types.is_integer_dtype(col_dtypes[col]):
            imputed_values = np.where(~np.isnan(imputed_values), np.round(imputed_values), np.nan)
            df.loc[null_indices[null_col_mask], col] = (
                pd.Series(imputed_values[null_col_mask.values], index=null_indices[null_col_mask])
                .astype("Int64")
            )
        else:
            df.loc[null_indices[null_col_mask], col] = imputed_values[null_col_mask.values].astype(col_dtypes[col])

    # Fiziksel sınırları tekrar kontrol ediyoruz
    bounds = {
        'Heart_Rate_(BPM)': (30, 200),
        'Blood_Oxygen_Level_(%)': (80, 100),
        'Step_Count': (0, 100000),
        'Sleep_Duration_(hours)': (0, 24),
        'Stress_Level': (1, 10)
    }

    for col, (min_val, max_val) in bounds.items():
        if col in df.columns:
            df[col] = df[col].clip(lower=min_val, upper=max_val)


# İmputasyonu uygula
custom_knn_imputation(df)

print("\nİşlem sonrası veri:")
print(df)

print(df.isnull().sum())


İşlem sonrası veri:
      Heart_Rate_(BPM)  Blood_Oxygen_Level_(%)  Step_Count  \
0            58.939776               98.809650        5450   
1            83.723932               98.532195         728   
2            81.067588               97.052954        2827   
3            69.620658               96.894213       13797   
4            61.950165               98.583797       15679   
...                ...                     ...         ...   
9995         78.819386               98.931927        2948   
9996         48.632659               95.773035        4726   
9997         73.834442               97.945874        2571   
9998         67.969560               98.401058        3365   
9999         70.063864               98.475606         545   

      Sleep_Duration_(hours) Activity_Level  Stress_Level  
0                   7.167236  Highly Active           1.0  
1                   6.538239  Highly_Active           5.0  
2                   5.026655  Highly Active           

In [240]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors

def fill_invalid_categorical(df, cat_col, numeric_cols, invalid_values=None, n_neighbors=3):

    invalid_values = invalid_values or ['', 'unknown', '??', None, np.nan]
    
    # Geçerli ve geçersiz değerleri belirle
    valid_mask = ~df[cat_col].isin(invalid_values) & df[cat_col].notnull()
    invalid_mask = ~valid_mask
    
    # Null varsa sayısal sütunları düşür, çünkü KNN bozulur
    df_numeric = df[numeric_cols].copy()
    df_numeric = df_numeric.dropna()

    # Geçerli verilerdeki kesişim
    valid_indices = df[valid_mask & df[numeric_cols].notnull().all(axis=1)].index
    invalid_indices = df[invalid_mask & df[numeric_cols].notnull().all(axis=1)].index

    if len(invalid_indices) == 0:
        print("Doldurulacak geçersiz kategori verisi bulunamadı.")
        return

    # Sayısal verileri ölçekle
    scaler = MinMaxScaler()
    X_valid = scaler.fit_transform(df.loc[valid_indices, numeric_cols])
    X_invalid = scaler.transform(df.loc[invalid_indices, numeric_cols])

    # KNN ile en yakın komşuları bul
    knn = NearestNeighbors(n_neighbors=n_neighbors)
    knn.fit(X_valid)
    distances, indices = knn.kneighbors(X_invalid)

    # Her geçersiz satır için en yaygın kategoriyi ata
    for i, row_idx in enumerate(invalid_indices):
        neighbor_idxs = valid_indices[indices[i]]
        most_common = df.loc[neighbor_idxs, cat_col].mode().iloc[0]
        df.at[row_idx, cat_col] = most_common

fill_invalid_categorical(
    df,
    cat_col='Activity_Level',
    numeric_cols=[
        'Heart_Rate_(BPM)', 
        'Blood_Oxygen_Level_(%)', 
        'Step_Count', 
        'Sleep_Duration_(hours)', 
        'Stress_Level'
    ],
    invalid_values=['', 'Actve', 'unknown']
)




In [241]:
print(df.isnull().sum())
df.head()

Heart_Rate_(BPM)          0
Blood_Oxygen_Level_(%)    0
Step_Count                0
Sleep_Duration_(hours)    0
Activity_Level            0
Stress_Level              0
dtype: int64


,Heart_Rate_(BPM),Blood_Oxygen_Level_(%),Step_Count,Sleep_Duration_(hours),Activity_Level,Stress_Level
0,58.939776,98.809650,5450,7.167236,Highly Active,1.0
1,83.723932,98.532195,728,6.538239,Highly_Active,5.0
2,81.067588,97.052954,2827,5.026655,Highly Active,5.0
3,69.620658,96.894213,13797,7.367790,Active,3.0
4,61.950165,98.583797,15679,6.400394,Highly_Active,6.0


In [242]:
print(df['Activity_Level'].unique())

['Highly Active' 'Highly_Active' 'Active' 'Seddentary' 'Sedentary']


In [243]:
# Bu satırlar için manuel düzeltme yapacağız
df['Activity_Level'] = df['Activity_Level'].replace({
    'Highly_Active': 'Highly Active',
    'Seddentary': 'Sedentary',
    'Actve':'Active'
})


In [244]:
print(df['Activity_Level'].unique())
print(df['Stress_Level'].unique())

['Highly Active' 'Active' 'Sedentary']
[ 1.          5.          3.          6.         10.          2.
  8.          9.          6.          4.          5.66666667  4.
  7.          6.          4.66666667  6.66666667  4.33333333  7.
  3.66666667  2.66666667  9.33333333  4.33333333  7.33333333  5.
  6.33333333  4.66666667  5.33333333  7.66666667  3.66666667  7.33333333
  7.66666667  5.33333333  2.33333333  8.66666667  3.          3.66666667
  3.33333333  6.33333333  8.33333333  8.33333333  4.66666667]


In [245]:
df['Stress_Level'] = df['Stress_Level'].round().astype(int)
print(df['Stress_Level'].unique())



[ 1  5  3  6 10  2  8  9  4  7]


In [246]:
print(df.isnull().sum())

Heart_Rate_(BPM)          0
Blood_Oxygen_Level_(%)    0
Step_Count                0
Sleep_Duration_(hours)    0
Activity_Level            0
Stress_Level              0
dtype: int64


In [247]:
df.head()

,Heart_Rate_(BPM),Blood_Oxygen_Level_(%),Step_Count,Sleep_Duration_(hours),Activity_Level,Stress_Level
0,58.939776,98.809650,5450,7.167236,Highly Active,1
1,83.723932,98.532195,728,6.538239,Highly Active,5
2,81.067588,97.052954,2827,5.026655,Highly Active,5
3,69.620658,96.894213,13797,7.367790,Active,3
4,61.950165,98.583797,15679,6.400394,Highly Active,6


In [248]:
import openpyxl

df.to_excel("dataset_clean.xlsx", index=False, engine='openpyxl')
